# 02 — Relationships Between Variables

Companion to [`../../data_visualization/relationships.md`](../../data_visualization/relationships.md).

---

## Why relationship plots matter

Understanding relationships between variables is the heart of exploratory data analysis. These plots answer:

- **Are two variables correlated?** (scatter, hexbin)
- **What's the strength and direction?** (correlation matrix)
- **Is the relationship linear or something more complex?** (regression lines, lowess)
- **Do patterns differ across groups?** (hue/faceting)
- **Are there confounding variables?** (partial plots, stratification)

All data used is drawn from the handbook's [`sample_data`](../sample_data/) directory with industry-relevant context.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.nonparametric.smoothers_lowess import lowess

sns.set_theme(style='whitegrid', font_scale=1.0)
rng = np.random.default_rng(42)

print('Libraries loaded.')

## 1. Scatter plot — the workhorse

Using `sample_data/relationships/scatterplot.csv` — **ad spend vs revenue** for different marketing algorithms.
This is a classic **marketing/analytics** scenario: does spending more on ads actually drive more revenue?

In [ ]:
scatter_data = pd.read_csv('../sample_data/relationships/scatterplot.csv')
scatter_data.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=scatter_data, x='ad_spend', y='revenue', hue='algorithm', 
                alpha=0.6, s=60, ax=ax, palette='Set2')

# Add regression line per algorithm
for alg in scatter_data['algorithm'].unique():
    subset = scatter_data[scatter_data['algorithm'] == alg]
    z = np.polyfit(subset['ad_spend'], subset['revenue'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(subset['ad_spend'].min(), subset['ad_spend'].max(), 100)
    ax.plot(x_line, p(x_line), '--', linewidth=1.5, alpha=0.7)

ax.set_xlabel('Ad Spend ($)'); ax.set_ylabel('Revenue ($)')
ax.set_title('Ad Spend vs Revenue by Algorithm\n(Marketing Analytics — "Does spending drive revenue?")')
ax.legend()
plt.tight_layout(); plt.show()

print('Correlation per algorithm:')
for alg in scatter_data['algorithm'].unique():
    subset = scatter_data[scatter_data['algorithm'] == alg]
    r = subset['ad_spend'].corr(subset['revenue'])
    print(f"  {alg}: r = {r:.3f}")

## 2. Bubble chart — adding a 4th dimension

Using `sample_data/relationships/bubble_chart.csv` — **global city data** with population, GDP, and region.
This is a **geospatial/economic** analysis: city size, wealth, and regional patterns.

In [ ]:
bubble_data = pd.read_csv('../sample_data/relationships/bubble_chart.csv')
bubble_data

In [ ]:
# Bubble chart: x=longitude, y=latitude, size=population, color=GDP, hue=region
fig, ax = plt.subplots(figsize=(12, 7))

palette = {'Asia': '#e74c3c', 'Europe': '#3498db', 'North America': '#2ecc71', 'South America': '#f39c12'}
for region in bubble_data['region'].unique():
    subset = bubble_data[bubble_data['region'] == region]
    # Scale bubble area proportional to population (not radius)
    areas = subset['population'] / 100
    ax.scatter(subset['longitude'], subset['latitude'], s=areas, 
               c=palette.get(region, 'gray'), alpha=0.6, label=region, edgecolors='white', linewidth=0.5)
    for _, row in subset.iterrows():
        ax.annotate(f"{row['region']} ({row['population']}M)", 
                   (row['longitude'], row['latitude']), 
                   textcoords='offset points', xytext=(5, 5), fontsize=7)

ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Global Cities — Population (size) & GDP (color)\n(Economic Geography)')
ax.legend(title='Region')
ax.set_xlim(-180, 180)
plt.tight_layout(); plt.show()

## 3. Overplotting — hexbin to the rescue

When you have thousands of points, scatter plots become a solid blob. Hexbin aggregates points into hexagonal bins.

In [ ]:
# Generate large dataset to demonstrate overplotting
n = 5000
x = rng.normal(0, 1, n)
y = 2 * x + rng.normal(0, 1, n)
large_df = pd.DataFrame({'x': x, 'y': y})

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Scatter with transparency (still overplotted)
axes[0].scatter(large_df['x'], large_df['y'], alpha=0.03, s=5, color='steelblue')
axes[0].set_title('Scatter (alpha=0.03)\n(Still a blob...)')

# Hexbin
hb = axes[1].hexbin(large_df['x'], large_df['y'], gridsize=30, cmap='viridis')
axes[1].set_title('Hexbin (30×30 grid)\n(Color = point density)')
plt.colorbar(hb, ax=axes[1], label='count')

# Hexbin with log color scale
hb2 = axes[2].hexbin(large_df['x'], large_df['y'], gridsize=40, cmap='plasma', mincnt=1)
axes[2].set_title('Hexbin (log color)\n(Better contrast)')
plt.colorbar(hb2, ax=axes[2], label='count')

plt.tight_layout(); plt.show()

## 4. 2D density contour

KDE contours show the density structure in 2D — like hexbin but with smooth boundaries.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2D KDE contour (filled)
sns.kdeplot(data=large_df, x='x', y='y', ax=axes[0], fill=True, cmap='viridis', levels=15)
axes[0].set_title('2D KDE contour (filled)\n(Smooth density estimate)')

# 2D KDE contour (line style)
sns.kdeplot(data=large_df, x='x', y='y', ax=axes[1], fill=False, cmap='viridis', levels=15)
axes[1].set_title('2D KDE contour (line)\n(Cleaner for presentations)')

plt.tight_layout(); plt.show()

## 5. Correlation matrix heatmap

A heatmap of pairwise correlations gives a quick overview of all relationships at once.

In [ ]:
# Generate a realistic correlation scenario
n = 500
data = {
    'ad_spend': rng.normal(100, 30, n),
    'website_visits': lambda x: 2*x + rng.normal(0, 20, n),
    'conversions': lambda v: 0.05*v + rng.normal(0, 3, n),
    'revenue': lambda c: 50*c + rng.normal(0, 100, n),
    'bounce_rate': lambda v: -0.5*v + rng.normal(50, 10, n),
}
corr_df = pd.DataFrame({
    'ad_spend': data['ad_spend'],
    'website_visits': data['website_visits'](data['ad_spend']),
    'conversions': data['conversions'](data['website_visits']),
    'revenue': data['revenue'](data['conversions']),
    'bounce_rate': data['bounce_rate'](data['website_visits']),
})

corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1, fmt='.3f',
            square=True, ax=ax, linewidths=1, linecolor='white',
            xticklabels=['Ad Spend', 'Visits', 'Conversions', 'Revenue', 'Bounce Rate'],
            yticklabels=['Ad Spend', 'Visits', 'Conversions', 'Revenue', 'Bounce Rate'])
ax.set_title('Marketing Funnel — Correlation Matrix\n(Which metrics drive revenue?)')
plt.tight_layout(); plt.show()

## 6. Scatter matrix (pairplot)

Shows all pairwise relationships at once — scatter plots off-diagonal, distributions on-diagonal.

In [ ]:
g = sns.pairplot(corr_df.sample(300), diag_kind='kde', palette='Set2')
g.fig.suptitle('Marketing Funnel — Pairplot\n(All pairwise relationships)', y=1.02)
plt.tight_layout(); plt.show()

## 7. LOWESS — non-parametric smooth

LOWESS reveals the true shape of the relationship without assuming linearity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear relationship (ad spend → revenue)
axes[0].scatter(corr_df['ad_spend'], corr_df['revenue'], alpha=0.1, s=20, color='steelblue', label='data')
smooth = lowess(corr_df['revenue'], corr_df['ad_spend'], frac=0.2)
axes[0].plot(smooth[:, 0], smooth[:, 1], 'r-', linewidth=3, label='LOWESS')
axes[0].set_xlabel('Ad Spend ($)'); axes[0].set_ylabel('Revenue ($)')
axes[0].set_title('Ad Spend vs Revenue — Linear\n(LOWESS ≈ straight line)')
axes[0].legend()

# Non-linear relationship (bounce rate → conversions)
axes[1].scatter(corr_df['bounce_rate'], corr_df['conversions'], alpha=0.1, s=20, color='steelblue', label='data')
smooth2 = lowess(corr_df['conversions'], corr_df['bounce_rate'], frac=0.2)
axes[1].plot(smooth2[:, 0], smooth2[:, 1], 'r-', linewidth=3, label='LOWESS')
axes[1].set_xlabel('Bounce Rate (%)'); axes[1].set_ylabel('Conversions')
axes[1].set_title('Bounce Rate vs Conversions — Non-linear\n(LOWESS reveals the curve)')
axes[1].legend()

plt.tight_layout(); plt.show()

## 8. Parallel coordinates — multivariate relationships

Using `sample_data/relationships/parallel_coordinates.csv` — comparing **laptop specifications and ratings**.
This is a **retail product analysis** scenario: which specs drive higher ratings?

In [ ]:
parallel_data = pd.read_csv('../sample_data/relationships/parallel_coordinates.csv')
parallel_data

In [ ]:
# Normalize each column to 0-1 scale for parallel coordinates
numeric_cols = ['price', 'ram_gb', 'storage_gb', 'battery_hours', 'rating']
normalized = parallel_data[numeric_cols].copy()
for col in numeric_cols:
    normalized[col] = (normalized[col] - normalized[col].min()) / (normalized[col].max() - normalized[col].min())

fig, ax = plt.subplots(figsize=(12, 6))

palette = {'Tech': '#2ecc71', 'Budget': '#e74c3c', 'Premium': '#3498db'}
for category, color in palette.items():
    subset = normalized[parallel_data['category'] == category]
    for _, row in subset.iterrows():
        ax.plot(range(len(numeric_cols)), row.values, color=color, alpha=0.3, linewidth=1)

ax.set_xticks(range(len(numeric_cols)))
ax.set_xticklabels([c.replace('_', '\n') for c in numeric_cols], fontsize=9)
ax.set_title('Parallel Coordinates — Laptop Specs by Category\n(Retail Product Analysis)')
ax.set_ylabel('Normalized (0–1)')
ax.legend(title='Category', loc='upper right')
plt.tight_layout(); plt.show()

## 9. Span chart — showing ranges

Using `sample_data/relationships/span_chart.csv` — **temperature ranges by month**.
This is a **climate/weather** analysis: daily temperature variation across the year.

In [ ]:
span_data = pd.read_csv('../sample_data/relationships/span_chart.csv')
span_data

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Horizontal span bars (floating bars)
for i, (_, row) in enumerate(span_data.iterrows()):
    ax.barh(i, row['high_temp'] - row['low_temp'], 
            left=row['low_temp'], height=0.5, color='steelblue', alpha=0.6, edgecolor='white')
    # Mark the mean (midpoint)
    mid = (row['low_temp'] + row['high_temp']) / 2
    ax.plot(mid, i, 'ro', markersize=8)

ax.set_yticks(range(len(span_data)))
ax.set_yticklabels(span_data['month'])
ax.set_xlabel('Temperature (°C)')
ax.set_title('Monthly Temperature Ranges — Low/High\n(Climate/Weather Analysis)\nRed dot = midpoint')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

## 10. Marginal scatter plot

Adds histograms/KDEs along the axes — showing both relationship and individual distributions.

In [ ]:
fig = plt.figure(figsize=(10, 8))

# Main scatter plot
ax_main = fig.add_axes([0.2, 0.15, 0.7, 0.75])
sns.scatterplot(data=corr_df.sample(300), x='ad_spend', y='revenue', ax=ax_main, 
                alpha=0.4, s=30, color='steelblue')
ax_main.set_xlabel('Ad Spend ($)'); ax_main.set_ylabel('Revenue ($)')
ax_main.set_title('Marginal Scatter Plot')

# Top marginal KDE
ax_top = fig.add_axes([0.2, 0.9, 0.7, 0.08])
sns.kdeplot(corr_df['ad_spend'], ax=ax_top, color='steelblue', alpha=0.7)
ax_top.axis('off')

# Right marginal KDE
ax_right = fig.add_axes([0.9, 0.15, 0.08, 0.75])
sns.kdeplot(corr_df['revenue'], ax=ax_right, color='steelblue', alpha=0.7, vertical=True)
ax_right.axis('off')

plt.show()

## Pitfalls

- **Correlation ≠ causation**: Two variables can be correlated through a hidden confounder.
- **Simpson's paradox**: A trend that appears in groups can reverse when you aggregate.
- **Overplotting hides structure**: Always check if your scatter plot is just a solid blob — use hexbin or 2D KDE instead.
- **t-SNE distances**: When you reduce dimensions with t-SNE / UMAP, **trust cluster membership, not distances between clusters**. The geometry is locally faithful but globally distorted.